In [4]:
import os
import random
import librosa
import numpy as np
import sounddevice as sd
from scipy.io.wavfile import write
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import load_model

# Import configurations
import sys
sys.path.append(os.path.abspath('..'))
from config import DATA_PATH, PROCESSED_DATA_PATH, MODEL_SAVE_PATH, SENTENCES, SAMPLE_RATE, DURATION, N_MFCC

In [5]:
print("Loading model...")
model = load_model(MODEL_SAVE_PATH)

# Load the raw training data features
X_train_raw = np.load(os.path.join(PROCESSED_DATA_PATH, 'X.npy'))

# Reshape to 2D exactly like in train.py (nsamples, nx * ny)
nsamples, nx, ny = X_train_raw.shape
X_train_raw = X_train_raw.reshape((nsamples, nx * ny))

# Fit the scaler
scaler = StandardScaler()
scaler.fit(X_train_raw)

# Setup Label Encoder to map numerical predictions back to text ('s01', 's02', etc.)
Y_raw = np.load(os.path.join(PROCESSED_DATA_PATH, 'Y.npy'))
le = LabelEncoder()
le.fit(Y_raw)

print("Model and Scaler ready!")

Loading model...
Model and Scaler ready!


In [6]:
import os
import librosa
import numpy as np
import joblib
from tensorflow.keras.models import load_model
from config import SAMPLE_RATE, DURATION, MODEL_SAVE_PATH

# --- CONFIG & CONSTANTS ---
# Ensure these match exactly what was used during training
N_MFCC = 20
FRAME_LENGTH = 2048
HOP_LENGTH = 512

# Pathing for the joblib files in your models folder
MODELS_DIR = os.path.dirname(MODEL_SAVE_PATH)
SCALER_PATH = os.path.join(MODELS_DIR, 'scaler.joblib')
LE_PATH = os.path.join(MODELS_DIR, 'label_encoder.joblib')

# Load the artifacts globaly so they are available to the functions
model = load_model(MODEL_SAVE_PATH)
scaler = joblib.load(SCALER_PATH)
le = joblib.load(LE_PATH)

def process_audio_array(audio, sample_rate):
    """Processes raw audio arrays to extract and stack features."""
    # 1. Padding / Truncation
    target_length = int(sample_rate * DURATION)
    if len(audio) < target_length:
        audio = np.pad(audio, (0, max(0, target_length - len(audio))), "constant")
    else:
        audio = audio[:target_length]
    
    # 2. Extract the 3 features (ZCR, RMS, MFCC)
    res1 = librosa.feature.zero_crossing_rate(y=audio, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH)
    res2 = librosa.feature.rms(y=audio, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH)
    res3 = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=N_MFCC, n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH)
    
    res1 = np.squeeze(res1)
    res2 = np.squeeze(res2)
    res3 = np.ravel(res3.T)
    
    # 3. Stack horizontally
    result = np.hstack((res1, res2, res3))
    return result

def extract_and_predict(file_path):
    """Extracts features from an audio file, scales them, and predicts the sentence."""
    # 1. Load Audio
    audio, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION, offset=0.0)
    
    # 2. Extract Features exactly as done in training
    features = process_audio_array(audio, sr)
    
    # 3. Reshape for Scaler (1, total_feature_count)
    features_reshaped = features.reshape(1, -1) 
    
    # 4. Scale Features using the loaded joblib scaler
    features_scaled = scaler.transform(features_reshaped)
    
    # 5. Predict
    prediction_probs = model.predict(features_scaled, verbose=0)
    predicted_class_idx = np.argmax(prediction_probs, axis=1)[0]
    
    # 6. Inverse transform the label using the loaded joblib encoder
    predicted_sentence = le.inverse_transform([predicted_class_idx])[0]
    confidence = np.max(prediction_probs) * 100
    
    return predicted_sentence, confidence

/Users/ayamkattel/Desktop/MISC_LATEST/Nepali-Speech-Recognition/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/ayamkattel/Desktop/MISC_LATEST/Nepali-Speech-Recognition/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.5.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
all_files = []
for root, _, files in os.walk(DATA_PATH):
    folder_name = os.path.basename(root) # Retain exact case
    if folder_name in SENTENCES:
        for file in files:
            if file.endswith('.wav'):
                all_files.append((os.path.join(root, file), folder_name))

# Pick a random file
random_file_path, true_sentence = random.choice(all_files)

print(f"Testing File: {os.path.basename(random_file_path)}")
print(f"True Sentence Group: {true_sentence}")

predicted_sentence, confidence = extract_and_predict(random_file_path)

print(f"Predicted Sentence Group: {predicted_sentence}")
print(f"Confidence: {confidence:.2f}%")

Testing File: spk05_s06.wav
True Sentence Group: TimiLai_Kasto_Chha
Predicted Sentence Group: TimiLai_Kasto_Chha
Confidence: 99.42%


In [8]:
import pyaudio
import numpy as np
import librosa
from config import SAMPLE_RATE, DURATION # Ensure these are imported

def test_mic_sentence():
    """
    Opens the microphone, records audio for the CORRECT duration, 
    and predicts sentence class.
    """
    CHUNK = 1024
    FORMAT = pyaudio.paFloat32
    CHANNELS = 1
    RATE = SAMPLE_RATE 
    RECORD_SECONDS = DURATION 
    
    p = pyaudio.PyAudio()
    
    print(f"--- Recording for {RECORD_SECONDS} seconds... ---")
    stream = p.open(format=FORMAT, channels=CHANNELS, rate=RATE,
                    input=True, frames_per_buffer=CHUNK)

    frames = []
    for i in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
        data = stream.read(CHUNK)
        frames.append(np.frombuffer(data, dtype=np.float32))

    print("--- Processing... ---")
    stream.stop_stream()
    stream.close()
    p.terminate()

    audio_data = np.concatenate(frames)

    # Use the unified process_audio_array logic from above to fix padding & stacking
    features = process_audio_array(audio_data, RATE)
    
    # Predict directly via unified extraction
    features_scaled = scaler.transform(features.reshape(1, -1))

    prediction_probs = model.predict(features_scaled, verbose=0)
    predicted_class_idx = np.argmax(prediction_probs, axis=1)[0]
    predicted_sentence = le.inverse_transform([predicted_class_idx])[0]
    confidence = np.max(prediction_probs) * 100

    print(f"Result: {predicted_sentence} ({confidence:.2f}%)")
    return predicted_sentence, confidence

In [52]:
test_mic_sentence()

--- Recording for 4.224 seconds... ---
--- Processing... ---
Result: Namaste (68.74%)


(np.str_('Namaste'), np.float32(68.742874))